In [1]:
from coil_fem.simsopt import CoilSupportBeamsCSRSorted, CoilFEMObjective
from simsopt import load, save
from pathlib import Path
import json
import math

ModuleNotFoundError: No module named 'coil_fem'

In [ ]:
fin_Jstress = load('fin_Jstress.json')[0]

In [ ]:
_OPTIONS_PATH = "../beam-options.json"
opts = json.load(open(_OPTIONS_PATH))
mesh_options = opts["mesh_options"]
material_options = opts["material_options"]
gravity_options = opts["gravity_options"]
problem_options = opts["problem_options"]
physics_options = opts["physics_options"]
beam_options = opts["beam_options"]
beam_options["n_beam_cr"] = 2
fixed_clamp_options = opts["fixed_clamp_options"]
mesh_scale = 0.5

# Circular CSR: R=4 m, section 0.3 x 0.5 m, Fourier order 2.
csr_order = 2
csr_r = 3.5
csr_options = {
    "order": csr_order,
    "w1": 0.3,   # width
    "w2": 0.5,   # height
    "n_phi": 64,
    "E": beam_options["E"],
    "nu": beam_options["nu"],
}

In [ ]:
fin_support = fin_Jstress.coil_support
dmin_csrcc = (
    0.5 * math.hypot(csr_options["w1"], csr_options["w2"])
    + 0.5 * math.hypot(mesh_options["w1"], mesh_options["w2"])
) * 1.2

In [ ]:
csr_support = CoilSupportBeamsCSRSorted.from_clamps(
    source=fin_support,
    coil_csr_distance=dmin_csrcc,
    csr_options=csr_options,
    problem_options=problem_options,
    thetas_orientation_cr=None,
    fixed_dof_names=None,
    n_ring_samples=200,
    r_beam = 0.08
    # **kwargs,
)

In [ ]:
Jstress_csr = CoilFEMObjective(
    csr_support,
    metrics          = ("sq_max_von_mises_lse"), # ("sq_max_von_mises_lse"), # ("l2_von_mises",),
    metric_weights   = (1.,),
    mesh_options     = mesh_options,
    material_options = material_options,
    gravity_options  = gravity_options,
    problem_options  = problem_options,
    physics_options  = physics_options,
    coupling         = "monolithic",
)

In [ ]:
Jstress_csr.save_support_vtu('init_support_csr')

In [ ]:
save([Jstress_csr], 'Jstress_csr.json')